## Genie Space Inventory Generation

This notebook discovers all Genie Spaces in the source workspace and generates an inventory CSV for review.

**What it does:**
1. Lists all Genie Spaces in the workspace
2. Extracts metadata (title, description, warehouse, benchmark count)
3. Writes inventory CSV to the export volume

**Prerequisites:**
- CAN_EDIT permission on Genie Spaces (to count benchmarks)
- READ/WRITE access to the export volume

## Install Dependencies

In [ ]:
%pip install databricks-sdk>=0.76.0 --quiet
dbutils.library.restartPython()

## Setup Path Resolution for Helpers

In [ ]:
import sys
import os

notebook_path = os.getcwd()
if notebook_path.startswith("/Workspace"):
    bundle_root = os.path.dirname(notebook_path)
else:
    bundle_root = os.path.dirname(os.path.abspath("__file__"))

helpers_path = os.path.join(bundle_root, "helpers")
if not os.path.exists(helpers_path):
    helpers_path = os.path.join(os.path.dirname(bundle_root), "helpers")

if helpers_path not in sys.path:
    sys.path.insert(0, os.path.dirname(helpers_path))

print(f"Bundle root: {bundle_root}")
print(f"Helpers path: {helpers_path}")

## Read Job Parameters

In [ ]:
dbutils.widgets.text("volume_path", "", "Export Volume Path")
dbutils.widgets.text("parent_path_filter", "/Workspace", "Parent Path Filter")

volume_path = dbutils.widgets.get("volume_path")
parent_path_filter = dbutils.widgets.get("parent_path_filter")

if not volume_path:
    raise ValueError("volume_path parameter is required")

print(f"Export volume: {volume_path}")
print(f"Parent path filter: {parent_path_filter}")

## Discover Genie Spaces

In [ ]:
from databricks.sdk import WorkspaceClient
from helpers.discovery import discover_genie_spaces_via_api, list_genie_spaces, filter_spaces_by_path

w = WorkspaceClient()
print(f"Connected to: {w.config.host}")

print("\nDiscovering Genie Spaces...")
spaces = discover_genie_spaces_via_api(w)

if not spaces:
    print("No spaces found via API, trying workspace listing...")
    spaces = list_genie_spaces(w, parent_path=parent_path_filter)

if parent_path_filter and parent_path_filter != "/Workspace":
    spaces = filter_spaces_by_path(spaces, parent_path_filter)

print(f"\nFound {len(spaces)} Genie Space(s)")

## Display Discovered Spaces

In [ ]:
import pandas as pd

if spaces:
    df = pd.DataFrame([s.to_dict() for s in spaces])
    display(df)
else:
    print("No Genie Spaces found in the specified path.")

## Extract catalog references from spaces

In [ ]:
import json

def extract_catalogs_from_serialized_space(serialized_space: str) -> set:
    """Extract catalog names from a serialized_space JSON."""
    catalogs = set()
    if not serialized_space:
        return catalogs
    
    try:
        data = json.loads(serialized_space)
        data_sources = data.get("data_sources", {})
        tables = data_sources.get("tables", [])
        
        for table in tables:
            identifier = table.get("identifier", "")
            if identifier:
                parts = identifier.split(".")
                if len(parts) >= 1:
                    catalogs.add(parts[0])
    except json.JSONDecodeError:
        pass
    
    return catalogs

space_catalogs = {}
all_catalogs = set()

print("Extracting catalog references...")
for space in spaces:
    try:
        space_data = w.genie.get_space(space_id=space.space_id, include_serialized_space=True)
        catalogs = extract_catalogs_from_serialized_space(space_data.serialized_space)
        space_catalogs[space.space_id] = catalogs
        all_catalogs.update(catalogs)
        print(f"  {space.title}: {catalogs if catalogs else 'No catalogs'}")
    except Exception as e:
        space_catalogs[space.space_id] = set()
        print(f"  {space.title}: Error - {e}")

print(f"\nUnique catalogs found: {sorted(all_catalogs)}")

## Write Inventory to Volume

In [ ]:
import csv

inventory_dir = os.path.join(volume_path, "genie_inventory")
os.makedirs(inventory_dir, exist_ok=True)

inventory_path = os.path.join(inventory_dir, "genie_inventory.csv")

fieldnames = ["space_id", "title", "description", "parent_path", "warehouse_id", "benchmark_count", "catalogs", "created_by", "created_at"]

with open(inventory_path, "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=fieldnames)
    writer.writeheader()
    for space in spaces:
        row = space.to_dict()
        row["catalogs"] = ",".join(sorted(space_catalogs.get(space.space_id, set())))
        writer.writerow(row)

print(f"\nInventory written to: {inventory_path}")
print(f"Total spaces: {len(spaces)}")

catalog_summary_path = os.path.join(inventory_dir, "catalog_summary.csv")
with open(catalog_summary_path, "w", newline="", encoding="utf-8") as f:
    writer = csv.writer(f)
    writer.writerow(["catalog", "spaces_using"])
    for cat in sorted(all_catalogs):
        spaces_using = [s.title for s in spaces if cat in space_catalogs.get(s.space_id, set())]
        writer.writerow([cat, "; ".join(spaces_using)])

print(f"Catalog summary written to: {catalog_summary_path}")

## Summary

In [ ]:
total_benchmarks = sum(s.benchmark_count for s in spaces)

print("=" * 60)
print("INVENTORY GENERATION COMPLETE")
print("=" * 60)
print(f"Spaces discovered: {len(spaces)}")
print(f"Total benchmarks:  {total_benchmarks}")
print(f"Inventory file:    {inventory_path}")
print("\nNext step: Open Src_02_Review_and_Approve.ipynb to review and approve spaces for export.")